<a href="https://colab.research.google.com/github/Dharshini11042004/Adaptive-Threat-Detection-in-Smart-Healthcare-Infrastructure-with-Agentic-AI-using-RL/blob/main/Agent2_IoT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# AGENT 2 : IOT BEHAVIOURAL SECURITY AGENT
# Autoencoder + PPO
!pip install gymnasium stable-baselines3 tensorflow scikit-learn pandas numpy -q
import os
import zipfile
import numpy as np
import pandas as pd
import gymnasium as gym
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
# STEP 1 : EXTRACT DATASET
zip_path = "/content/preprocessed_Datasets.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content")
print("Dataset extracted")
iot_path = "/content/preprocessed_Datasets/IoT/"
files = os.listdir(iot_path)
print("IoT files:", files)
# STEP 2 : LOAD DATA
dfs = []
for f in files:
    if f.endswith(".csv"):
        df = pd.read_csv(iot_path + f)
        dfs.append(df)
data = pd.concat(dfs, ignore_index=True)
print("Total dataset shape:", data.shape)
# STEP 3 : SAFE DATA CLEANING
data = data.replace([np.inf, -np.inf], np.nan)
for col in data.columns:
    if data[col].dtype == "object":
        data[col] = pd.factorize(data[col])[0]
data = data.fillna(data.median(numeric_only=True))
print("Clean dataset shape:", data.shape)
X = data.values.astype(np.float32)
# STEP 4 : NORMALIZE
scaler = StandardScaler()
X = scaler.fit_transform(X)
# STEP 5 : TRAIN TEST SPLIT
X_train, X_test = train_test_split(
    X,
    test_size=0.3,
    random_state=42
)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
feature_count = X_train.shape[1]
# STEP 6 : AUTOENCODER
inputs = layers.Input(shape=(feature_count,))
x = layers.Dense(64, activation="relu")(inputs)
x = layers.Dense(32, activation="relu")(x)
encoded = layers.Dense(16, activation="relu")(x)
x = layers.Dense(32, activation="relu")(encoded)
x = layers.Dense(64, activation="relu")(x)
decoded = layers.Dense(feature_count)(x)
model = Model(inputs, decoded)
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="mse"
)
# STEP 7 : TRAIN AUTOENCODER
print("\nTraining Autoencoder")
model.fit(
    X_train,
    X_train,
    epochs=20,
    batch_size=256,
    shuffle=True,
    verbose=1
)
# STEP 8 : PRECOMPUTE ANOMALY SCORES
train_recon = model.predict(X_train, verbose=0)
test_recon = model.predict(X_test, verbose=0)
train_errors = np.mean((X_train - train_recon)**2, axis=1)
test_errors = np.mean((X_test - test_recon)**2, axis=1)
threshold = train_errors.mean() + 2*train_errors.std()
print("Anomaly threshold:", threshold)
# STEP 9 : RL ENVIRONMENT (UNCHANGED)
class IoTEnv(gym.Env):
    def __init__(self, data, errors):
        super(IoTEnv, self).__init__()
        self.data = data
        self.errors = errors
        self.index = 0

        self.action_space = gym.spaces.Discrete(3)

        self.observation_space = gym.spaces.Box(
            low=-5,
            high=5,
            shape=(feature_count,),
            dtype=np.float32
        )
    def reset(self, seed=None, options=None):
        self.index = 0
        return self.data[self.index], {}
    def step(self, action):
        error = self.errors[self.index]
        if error < threshold:
            reward = 2 if action == 0 else -1
        elif error < threshold*1.5:
            reward = 2 if action == 1 else -1
        else:
            reward = 2 if action == 2 else -1
        self.index += 1
        done = self.index >= len(self.data)
        next_state = self.data[self.index] if not done else self.data[-1]
        return next_state, reward, done, False, {}
# STEP 10 : VECTOR ENVIRONMENT
env = DummyVecEnv([lambda: IoTEnv(X_train, train_errors)])
# STEP 11 : PPO AGENT
agent = PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=256,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    max_grad_norm=0.5,
    n_epochs=8,
    verbose=1
)
print("\nTraining PPO Agent")
agent.learn(total_timesteps=25000)
print("Training Completed")
# SAVE TRAIN + TEST OUTPUTS
records = []
def get_label(error):
    if error < threshold:
        return "Normal"
    elif error < threshold*1.5:
        return "Suspicious"
    else:
        return "Malicious"
def get_action_name(action):
    return ["Allow", "Monitor", "Isolate"][action]
# TRAIN DATA OUTPUT
for i, row in enumerate(X_train):
    action, _ = agent.predict(row)
    error = train_errors[i]
    records.append([
        "TRAIN",
        i,
        float(error),
        float(abs(error)*100),
        get_label(error),
        int(action),
        get_action_name(action)
    ])
# TEST DATA OUTPUT + METRICS
allow = monitor = isolate = correct = 0
for i, row in enumerate(X_test):
    action, _ = agent.predict(row)
    error = test_errors[i]
    true = 0 if error < threshold else 1 if error < threshold*1.5 else 2
    if action == true:
        correct += 1
    if action == 0:
        allow += 1
    elif action == 1:
        monitor += 1
    else:
        isolate += 1
    records.append([
        "TEST",
        i,
        float(error),
        float(abs(error)*100),
        get_label(error),
        int(action),
        get_action_name(action)
    ])
accuracy = (correct / len(X_test)) * 100
# SAVE CSV
output_df = pd.DataFrame(records, columns=[
    "Dataset",
    "Index",
    "Anomaly Score",
    "Risk Score",
    "Behaviour",
    "Action",
    "Decision"
])
save_path = "/content/drive/MyDrive/MultiAgentModels/agent_2_outputs.csv"
output_df.to_csv(save_path, index=False)
print("\nSaved → agent_2_outputs.csv")
# RESULTS
print("\n------------------")
print("AGENT 2 RESULT")
print("-----------------")
print("Traffic analysed:", len(X_test))
print("Allow:", allow)
print("Monitor:", monitor)
print("Isolate:", isolate)
print("\nAccuracy:", round(accuracy,2), "%")
# SAMPLE OUTPUT
print("\nSample Predictions\n")
for i in range(5):
    action,_ = agent.predict(X_test[i])
    risk_score = float(abs(test_errors[i])*100)
    if action == 0:
        behaviour="Normal"
        mitigation="Allow IoT device"
    elif action == 1:
        behaviour="Suspicious"
        mitigation="Monitor IoT device"
    else:
        behaviour="Malicious"
        mitigation="Isolate IoT device"
    print("Traffic Risk Score:", round(risk_score,2))
    print("Behaviour:", behaviour)
    print("Mitigation:", mitigation)
    print("---")

Dataset extracted
IoT files: ['Train_Test_IoT_Modbus_test.csv', 'Train_Test_IoT_Modbus_train.csv', 'Train_Test_IoT_Thermostat_test.csv', 'Train_Test_IoT_Thermostat_train.csv']
Total dataset shape: (63880, 10)
Clean dataset shape: (63880, 10)
Train shape: (44716, 10)
Test shape: (19164, 10)

Training Autoencoder
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.3185
Epoch 2/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0129
Epoch 3/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0065
Epoch 4/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0044
Epoch 5/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0033
Epoch 6/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0027
Epoch 7/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0023
Epoch 8/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0019
Epoch 9/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0016
Epoch 10/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0014
Epoch 11/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0012
Epoch 12/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0011
Epoch 13/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 9.9024e-04
Epoch 14/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 8.8934e-04
Epoch 15/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: